In [ ]:
import mysql.connector
from model_manager import ModelManager
import os
import json
from pathlib import Path
from data_manager import DataManager

os.environ["SQL_PW"] = "YOUR_PASSWORD"

# Connect to MySQL server
mydb = mysql.connector.connect(
    host="localhost", 
    user="root", 
    password="YOUR_PASSWORD", 
)

cursor = mydb.cursor()

# Choose relevant database
cursor.execute("USE `ts-predict-db`;")

In [ ]:
table_scripts = [
    """
    CREATE TABLE IF NOT EXISTS characterisation (
        reaction_type_id INT AUTO_INCREMENT PRIMARY KEY,
        reaction_type VARCHAR(50)
    );
    """,

    """
    CREATE TABLE IF NOT EXISTS source (
        source_id INT AUTO_INCREMENT PRIMARY KEY,
        source_url VARCHAR(200),
        source_method VARCHAR(200)
    );
    """,

    """
    CREATE TABLE IF NOT EXISTS reaction (
        reaction_id INT AUTO_INCREMENT PRIMARY KEY,
        reactant JSON,
        product JSON,
        reactant_complex JSON,
        product_complex JSON,
        transition_state JSON,
        reaction_type_id INT,
        source_id INT,
        FOREIGN KEY (reaction_type_id)
            REFERENCES characterisation(reaction_type_id)
            ON DELETE SET NULL,
        FOREIGN KEY (source_id)
            REFERENCES source(source_id)
            ON DELETE SET NULL
    );
    """
]

# Create tables in database
for script in table_scripts:
    cursor.execute(script)

In [5]:
# Insert sources to source table
cursor.execute("INSERT INTO source (source_url, source_method) VALUES('https://doi.org/10.1088/2632-2153/aba822', 'MP2/6-311G(d)');")
cursor.execute("INSERT INTO source (source_url, source_method) VALUES('https://doi.org/10.1039/D1SC01206A', 'MP2/cc-PVDZ');")
cursor.execute("INSERT INTO source (source_url, source_method) VALUES('https://doi.org/10.1038/s41597-020-0460-4', 'ωB97X-D3/def2-TZVP');")
cursor.execute("INSERT INTO source (source_url, source_method) VALUES('https://doi.org/10.6084/m9.figshare.19614657.v4', 'ωB97x/6-31G(d)');")

In [6]:
# Insert characterisations to characterisation table
cursor.execute("INSERT INTO characterisation (reaction_type) VALUES('bimolecular nucleophilic substitution (SN2)')")
cursor.execute("INSERT INTO characterisation (reaction_type) VALUES('bimolecular elimination (E2)')")
cursor.execute("INSERT INTO characterisation (reaction_type) VALUES('isomerisation')")
cursor.execute("INSERT INTO characterisation (reaction_type) VALUES('unspecified')")

In [7]:
# Parse datasets
riley_data = DataManager.han_hdf5_file_parser(Path("data/riley_dataset.hdf5"))
isomerisation_data = DataManager.han_hdf5_file_parser(Path("data/isomerization_dataset.hdf5"))
transition1x_data = DataManager.transition1x_hdf5_file_parser(Path("data/transition1x_dataset.h5"))

In [8]:
# Insert reactions into reaction table
DataManager.sql_insert_reactions(riley_data, 1, 2, cursor)
DataManager.sql_insert_reactions(isomerisation_data, 3, 3, cursor)
DataManager.sql_insert_reactions(transition1x_data, 4, 4, cursor)

In [4]:
# For testing
cursor.execute("delete from reaction")
cursor.execute("ALTER TABLE reaction AUTO_INCREMENT = 1")
cursor.execute("delete from source")
cursor.execute("ALTER TABLE source AUTO_INCREMENT = 1")
cursor.execute("delete from characterisation")
cursor.execute("ALTER TABLE characterisation AUTO_INCREMENT = 1")

In [ ]:
cursor.execute("SELECT * FROM reaction;")
reactions = cursor.fetchall()
columns = [col[0] for col in cursor.description]

reaction_dicts = DataManager.sql_parse_reactions(reactions, columns)

# Predict transition state for each reaction and corresponding error
for reaction in reaction_dicts:
    predicted_ts = ModelManager.interpolate_model(reaction)
    error = ModelManager.error(reaction, predicted_ts)

0.5500645422654692
5.445082202468347
1.6689155367410913
1.9992404567434248
0.8552697815312269
0.8117111186632324
1.8358001811307387
0.47577653434878664
0.31676860172308696
0.5922200254692829
0.3167555005085829
21.70788918241447
25.48227010409391
10.140182173242358
15.520683224040644
1.1200733286340465
0.4448088624336394
0.5542739203092092
12.04343303423018
0.316815549735587
17.46726839617492
1.2046876406353437
4.547974749471748
8.584630537701871
2.4314854708673774
1.767199939731655
1.9221971141347671
2.748513481488981
3.6392472751108755
8.111940814202626
2.8411256905564555
0.5500279353930423
6.230486663633997
17.21803019688785
12.816259569788198
12.006754891700375
3.2672018670075986
2.414761493425215
0.44181965715391447
24.844956943990457
18.729799883404905
19.471555504809274
6.481621176528745
3.77586832868282
2.183913170527984
2.403247849183632
15.372022064364089
0.44178316935620715
17.05999369251315
12.79917166526686
10.850439041334624
7.574264290815346
7.588581595694219
0.5880204522

In [6]:
cursor.close()

True